associate_boxes_ADCs.py -- James Sayre, jsayre@berkeley.edu

Associate ADCs across the older and 2016 shapefiles as well as to box IDs in satellite imagery.

Also aggregates 2007 ADC Census info to 2016 ADCs as reference.

01/31/2024 -- May want to associate ADCs between 2007 and 2016 based on intersection with SIAP agland map!


In [ ]:
### Programs
import os, sys
os.environ['USE_PYGEOS'] = '0'
import geopandas as gpd
import shapely
import xarray as xr
from shapely.geometry import Polygon
from shapely.geometry import Point
from shapely.ops import cascaded_union

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from packaging import version

def create_box_geodf(geodf, delta = 0.008333333):
    ### The file "mun_boxes_'+ciclo+'_start_'+yr+'.csv" tells us about all of the relevant AOIs/boxes for a given
    ### municipality. In this file, each "box" is just a lat/lon coordinate representing the center of the box. 
    ### What we need to subset down imagery to each box is to create a geopandas dataframe where the box 
    ### dimensions are a polygon. This returns a polygon of said box.
    ### xcenter, ycenter -- lat/lon representing center of the box
    ### delta -- the height/weight of boxes, don't change this unless you change the size of boxes in 
    ### create_suitability_index.py
    
    xcenter, ycenter = geodf['x'], geodf['y']
    ytop, ybottom = ycenter+(delta/2.0), ycenter-(delta/2.0) 
    xleft, xright = xcenter-(delta/2.0), xcenter+(delta/2.0)
    box_poly = Polygon([(xleft, ytop), (xright, ytop), (xright, ybottom), (xleft, ybottom)])
    box_poly_inv = Polygon([(ytop, xleft), (ytop, xright), (ybottom, xright), (ybottom, xleft)])
    
    ### We need to account for how geopandas flips indices between version 0.5 and 0.6
    if version.parse(gpd.__version__) >= version.parse("0.6"):
        return box_poly
#         poly_df = gpd.GeoDataFrame({'geometry':[box_poly]})
#         poly_df.crs = {'init' :'EPSG:4326'}
    else:
        return box_poly_inv
#         poly_df = gpd.GeoDataFrame({'geometry':[box_poly_inv]})
#         poly_df.crs = "EPSG:4326"
    
#     poly_df['muncode'] = str(municipality_code)
#     return poly_df, ytop, ybottom, xleft, xright

### Directories
topdir             =  "/home/j/Dropbox/Projects/"
projectdir         =  os.path.join(topdir, "Maize_prediction")
cropdir            =  os.path.join(topdir, "Crop_misallocation")
sciagadir          =  os.path.join(cropdir,"data","SCIAGA")
datadir            =  os.path.join(projectdir,"Data")
amcadir            =  os.path.join(datadir,"INEGI","Areas_Censal_Agropecuario_2016")
agland_dir         =  os.path.join(datadir, "Mexico_agland","Inputs")
siap_agland_dir    =  os.path.join(datadir, "SIAP_agland","Output")
inegi_md_dir       =  os.path.join(datadir,"INEGI","MD_lab_outputs")
census_dir         =  os.path.join(inegi_md_dir,"CA_ADC_data_2023-05-2023")
amca_data_dir      =  os.path.join(inegi_md_dir,"AMCA_ADC_data_2022-11-23")

### Inputs
adcshp             =  os.path.join(os.path.expanduser("~"), "Dropbox", "Projects", "Maize_prediction", "Data", "Shapefiles", "adc_shapefile.shp") ###227,080 adcs in mdadc07, 295,184 here, 215,749 with yields
amca2016           =  os.path.join(amcadir,"census_areas.shp")
crop_corr          =  os.path.join(cropdir,"data","commodity_correspondence_table.csv")
agland_path        =  os.path.join(agland_dir, "agland_udsIV.shp")
siap_agland        =  os.path.join(siap_agland_dir, "agland_cader.shp")
ca2007_data        =  os.path.join(census_dir, "rendimiento_agr_adc.dta")
amca16_data        =  os.path.join(amca_data_dir, "amca_sup_adc.csv")

### Outputs 
amca_bid_out_file  =  os.path.join(datadir,"corr_tables","bid_amca_corr.csv")
ca07_bid_out_file  =  os.path.join(datadir,"corr_tables","bid_ca07_corr.csv")
ca07_ca16_out_file =  os.path.join(datadir,"corr_tables","ca07_ca16_corr.dta")
ca16_ca07_out_file =  os.path.join(datadir,"corr_tables","ca16_ca07_corr.dta")
ca07_ca16_inegi_ag =  os.path.join(datadir,"corr_tables","ca07_ca16_corr_inegi_agmask.dta")
ca16_ca07_inegi_ag =  os.path.join(datadir,"corr_tables","ca16_ca07_corr_inegi_agmask.dta")
ca07_ca16_siap_ag  =  os.path.join(datadir,"corr_tables","ca07_ca16_corr_siap_agmask.dta")
ca16_ca07_siap_ag  =  os.path.join(datadir,"corr_tables","ca16_ca07_corr_siap_agmask.dta")


ca2007_maize_fl    =  os.path.join(inegi_md_dir,"ca2007_maize_amca_adcs.dta")
ca2007_avocados_fl =  os.path.join(inegi_md_dir,"ca2007_avocados_amca_adcs.dta")
amca_avocados_fl   =  os.path.join(inegi_md_dir,"amca2016_avocados_amca_adcs.dta")

# ca07_ca16_out_file_bycrop = os.path.join(datadir,"corr_tables","ca07_ca16_corr_bycrop.dta")


search_region_satelite_imgs = os.path.join(projectdir,"search_regions_img_corr.csv")

In [12]:
### Read in satelite image id information
search_df = pd.read_csv(search_region_satelite_imgs)
# search_df['geometry'] = search_df.apply(create_box_geodf,axis=1)
# search_df = gpd.GeoDataFrame(search_df)
# search_df = search_df.set_crs(4326)
# search_df = search_df[['bid','geometry']]

In [2]:
### Read in 2007 Area de Control information
adc07_df = gpd.read_file(adcshp)
adc07_df['id'] = adc07_df['adcid'].fillna(adc07_df['locid'])
adc07_df = adc07_df[['id','tablaAlias','geometry']]
adc07_df.columns = ['adc07','adc07_type','geometry']
adc07_df.set_geometry('geometry', inplace=True)

### Join searchdf with ADC 07
# joindf = gpd.sjoin(search_df, adc07_df, how='left', op='intersects')
# joindf = joindf.groupby('bid').agg(list).reset_index()[['bid','adc07','adc07_type']]
# joindf.to_csv(ca07_bid_out_file,index=False)

In [5]:
adc07_df['ha'] = adc07_df.to_crs(epsg=6372).geometry.area/10000.0

In [28]:
len(adc16_df[adc16_df['ha'] < 221])/len(adc16_df)

0.46993817489597545

In [4]:
### Read in AMCA 2016 ADC info
adc16_df = gpd.read_file(amca2016)
adc16_df = adc16_df[['CONTROL','geometry']]
adc16_df.columns = ['adc16','geometry']
# adc16_df['cve_ent'] = adc16_df['adc16'].apply(lambda x: x[:2])
adc16_df.set_geometry('geometry', inplace=True)
adc16_df = adc16_df.set_crs(adc07_df.crs)

### Join searchdf with ADC 16
# join16df = gpd.sjoin(search_df, adc16_df, how='left', op='intersects')
# join16df = join16df.groupby('bid').agg(list).reset_index()[['bid','adc16']]
# join16df.to_csv(amca_bid_out_file,index=False)

In [6]:
adc16_df['ha'] = adc16_df.to_crs(epsg=6372).geometry.area/10000.0

In [ ]:
#### 11/21/22: No reason to do this
### Combine the two ADC files
# joindf   = pd.read_csv(ca07_bid_out_file)[['bid','adc07']]
# join16df = pd.read_csv(amca_bid_out_file)
# bid_adc_corr_df = joindf.merge(join16df,on='bid')
# bid_adc_corr_df['adc07'] = bid_adc_corr_df['adc07'].apply(lambda x: x[2:-2].replace("', '","','").split("','"))
# bid_adc_corr_df['adc16'] = bid_adc_corr_df['adc16'].apply(lambda x: x[2:-2].replace("', '","','").split("','"))

In [16]:
adc16_use_df = pd.read_stata(ca16_ca07_out_file)
print("Len of ADC shapefiles", len(adc07_df), len(adc16_df))
print("Number of ADCs in merged files", len(adc16_use_df['adc07'].unique()), len(adc16_use_df['adc16'].unique()))


Len of ADC shapefiles 299363 178083
Number of ADCs in merged files 298570 178021


In [20]:
### Compute agricultural land area for each ADC
agland_df = gpd.read_file(agland_path)

### Agland map might change between years, not going to use
### Merge with 2007 ADC
intersections = gpd.overlay(agland_df, adc07_df, how='intersection')
# intersections['adc07_agland_area'] = intersections.to_crs(epsg=6372).geometry.area/10000.0
# intersections = intersections.groupby('adc07')['adc07_agland_area'].sum().reset_index()
# adc07_df = adc07_df.merge(intersections, on='adc07', how='left')

KeyboardInterrupt: 

In [65]:
typ_yr_to_yr = '2007' ### 2016
use_agland   = 'INEGI' ### or 'SIAP' 

### Now compute size of intersection between adc07 and adc16
if typ_yr_to_yr == '2016':
    adc_df = gpd.sjoin(adc16_df, adc07_df, how='left', op='intersects')
    adc07_m_df = adc07_df[['adc07','geometry']]
    adc07_m_df.columns = ['adc07','geometry2']
    adc_df = adc_df.merge(adc07_m_df, on = 'adc07', how='left')
    
else:
    adc_df = gpd.sjoin(adc07_df, adc16_df, how='left', op='intersects')
    adc16_m_df = adc16_df[['adc16','geometry']]
    adc16_m_df.columns = ['adc16','geometry2']
    adc_df = adc_df.merge(adc16_m_df, on = 'adc16', how='left')
adc_df.drop(['index_right','adc07_type'],axis=1,inplace=True)

### Compute area of intersection between ADC16 and ADC07

### Note: doing area calcs in WGS84 is not correct, but because ADCs are so small, likely not a big distortion
def find_int_area(dataframe):
    try: 
        return dataframe['geometry'].intersection(dataframe['geometry2']).area
    except:
        return 0.0
    
adc_df['int_area'] = adc_df.apply(lambda row: find_int_area(row), axis = 1)
adc_df.drop(['geometry','geometry2'], axis=1, inplace=True)
if typ_yr_to_yr == '2016':
    adc_total_area = adc_df.groupby('adc16')['int_area'].sum().reset_index()
    adc_total_area.columns = ['adc16','adc16_total_area']
    adc_df = adc_df.merge(adc_total_area, on='adc16', how='left')
    adc_df['adc16share'] = adc_df['int_area']/adc_df['adc16_total_area']
    adc_df = adc_df[adc_df['adc16share'] >= 0.001]
    adc_df.drop(['adc16_total_area','adc16share'], axis=1, inplace=True)
    adc_total_area = adc_df.groupby('adc16')['int_area'].sum().reset_index()
    adc_total_area.columns = ['adc16','adc16_total_area']
    adc_df = adc_df.merge(adc_total_area, on='adc16', how='left')
    adc_df['adc16share'] = adc_df['int_area']/adc_df['adc16_total_area']
    adc_df.drop(['int_area','adc16_total_area'], axis=1, inplace=True)
    adc_df.to_stata(ca07_ca16_out_file, write_index=False)
else:
    adc_total_area = adc_df.groupby('adc07')['int_area'].sum().reset_index()
    adc_total_area.columns = ['adc07','adc07_total_area']
    adc_df = adc_df.merge(adc_total_area, on='adc07', how='left')
    adc_df['adc07share'] = adc_df['int_area']/adc_df['adc07_total_area']
    adc_df = adc_df[adc_df['adc07share'] >= 0.001]
    adc_df.drop(['adc07_total_area','adc07share'], axis=1, inplace=True)
    adc_total_area = adc_df.groupby('adc07')['int_area'].sum().reset_index()
    adc_total_area.columns = ['adc07','adc07_total_area']
    adc_df = adc_df.merge(adc_total_area, on='adc07', how='left')
    adc_df['adc07share'] = adc_df['int_area']/adc_df['adc07_total_area']
    adc_df.drop(['int_area','adc07_total_area'], axis=1, inplace=True)
    adc_df.to_stata(ca16_ca07_out_file, write_index=False)

### Note that both ADC07 and ADC16 ids are unique in each shapefile
### Drop any ADC that overlaps by less than 1/1000th (not likely to be super impt.)
    

/home/j/anaconda3/envs/geopd_env/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3382: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if await self.run_code(code, result, async_=asy):


In [39]:
## Now compute avocado/maize information in 2007 at 2016 ADC level
adc16_use_df = pd.read_stata(ca16_ca07_out_file)

ca07df = pd.read_stata(ca2007_data)[['adc','type',  'name','sup_sem', 'Q', 'num_terrenos']]
ca07df = ca07df.rename(columns={'adc':'adc07'})
### Subset down to only avocados/maize
ca07df       = ca07df[ca07df['name'] == 'Maize'].drop('name',axis=1)
ca07_oi_df   = ca07df[ca07df['type'] == 'o-i']
ca07_pv_df   = ca07df[ca07df['type'] == 'p-v']
ca07_tot_df  = ca07df.groupby('adc07')[['sup_sem','Q','num_terrenos']].sum().reset_index()
ca07_tot_df['type'] = 'total'

adc16_oi_df  = adc16_use_df.merge(ca07_oi_df, on='adc07', how='inner')
adc16_pv_df  = adc16_use_df.merge(ca07_pv_df, on='adc07', how='inner')
adc16_tot_df = adc16_use_df.merge(ca07_tot_df, on='adc07', how='inner')

adc16_use_df = pd.concat([adc16_oi_df,adc16_pv_df,adc16_tot_df])

### Avocados 
# ca07df = ca07df[ca07df['name'] == 'Avocados'].groupby('adc07')[['sup_sem','Q','num_terrenos']].sum().reset_index()
# ca07df['type'] = 'total'
# adc16_use_df = adc16_use_df.merge(ca07df, on='adc07', how='inner')

adc16_use_df['num_terrenos'] =  adc16_use_df['num_terrenos']*adc16_use_df['adc07share']
adc16_use_df['sup_sem']      =  adc16_use_df['sup_sem']*adc16_use_df['adc07share']
adc16_use_df['Q']            =  adc16_use_df['Q']*adc16_use_df['adc07share']

adc16_use_df = adc16_use_df.groupby(['adc16','type'])[['sup_sem', 'Q', 'num_terrenos']].sum().reset_index()
cycle_to_num_dict     = {'peren':3, 'o-i':1, 'p-v':2,'total':0}
adc16_use_df['cycle'] = adc16_use_df['type'].apply(lambda x: cycle_to_num_dict[x])
adc16_use_df['num_terrenos'] = adc16_use_df['num_terrenos'].apply(round)
adc16_use_df['num_terrenos'] = adc16_use_df['num_terrenos'].apply(lambda x: '*' if x < 3 else str(x))
adc16_use_df['yield'] = adc16_use_df['Q']/adc16_use_df['sup_sem']
adc16_use_df['name'] = 'Maize'
# adc16_use_df['name'] = 'Avocados'
adc16_use_df['year'] = 2007
adc16_use_df['muncode'] = adc16_use_df['adc16'].apply(lambda x: x[:5])
adc16_use_df['cve_ent'] = adc16_use_df['adc16'].apply(lambda x: x[:2])
adc16_use_df['ageb'] = adc16_use_df['adc16'].apply(lambda x: x[:10])
adc16_use_df.rename(columns={'adc16':'adc'},inplace=True)
# adc16_use_df.to_stata(ca2007_avocados_fl, write_index=False)
adc16_use_df.to_stata(ca2007_maize_fl, write_index=False)

### crop_name_df = pd.read_csv(crop_corr)[['name']]
### crops = list(set(crop_name_df['name'].to_list()))

### adc16corr_bycrop = pd.DataFrame()
### for crop in crops:
###    adc16_df['name'] = crop
###     adc16corr_bycrop = pd.concat([adc16corr_bycrop,adc16_df],axis=0)

### adc16corr_bycrop.to_stata(ca07_ca16_out_file_bycrop, write_index=False)

In [133]:
### Read in AMCA data but only for avocados
amca16_df = pd.read_csv(amca16_data)
amca16_df = amca16_df[amca16_df['name'] == 'Avocados']
amca16_df = amca16_df.drop(['super_cart','sup_irrig','sup_rf','yield_equiv','yield_gpw','yield_07w_equiv','yield_07w_gpw','yield_07yw_equiv','yield_07yw_gpw'], axis =1)
amca16_df['year'] = 2016
amca16_df.to_stata(amca_avocados_fl, write_index=False)


In [15]:
amca16_df = pd.read_csv(amca16_data)
amca16_df['num_terrenos'] = amca16_df['num_terrenos'].apply(lambda x: 1.5 if x == '*' else int(x))
amca16_df = amca16_df.groupby('adc')['num_terrenos'].sum().reset_index()